In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = r'D:\Naveed\Educational\retail-demand-forecasting\retail-demand-forecasting\data\processed\\'

In [2]:
RAW_PATH = r'D:\Naveed\Educational\retail-demand-forecasting\retail-demand-forecasting\data\raw\\'

sales_raw = pd.read_csv(RAW_PATH + 'sales_train_validation.csv')

sales_raw = sales_raw[(sales_raw['store_id'] == 'CA_1') & 
                       (sales_raw['cat_id'] == 'FOODS')]

print(sales_raw.shape)
sales_raw.head()

(1437, 1919)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
1612,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,0,0,1,...,0,2,0,4,1,1,0,1,1,0
1613,FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,0,1,0,1,...,1,3,1,0,0,1,2,0,0,0
1614,FOODS_1_003_CA_1_validation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,0,0,0,0,...,3,0,2,1,1,0,1,0,1,0
1615,FOODS_1_004_CA_1_validation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1616,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,3,9,3,3,...,3,1,1,2,0,2,2,1,4,1


In [3]:
id_cols = ['id','item_id','dept_id','cat_id','store_id','state_id']
day_cols = [c for c in sales_raw.columns if c.startswith('d_')]

sales = sales_raw.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)
print(sales.shape)

(2748981, 8)


In [4]:
calendar = pd.read_csv(RAW_PATH + 'calendar.csv', parse_dates=['date'])
sales = sales.merge(
    calendar[['d','date','event_name_1','snap_CA']], 
    on='d', how='left'
)
print(sales.shape)
sales.head()

(2748981, 11)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,event_name_1,snap_CA
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,NaN,0
1,FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0
2,FOODS_1_003_CA_1_validation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0
3,FOODS_1_004_CA_1_validation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0
4,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,NaN,0


In [5]:
sales['year'] = sales['date'].dt.year
sales['month'] = sales['date'].dt.month
sales['week'] = sales['date'].dt.isocalendar().week.astype(int)
sales['dayofweek'] = sales['date'].dt.dayofweek
sales['quarter'] = sales['date'].dt.quarter
sales['is_weekend'] = sales['dayofweek'].isin([5,6]).astype(int)
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,event_name_1,snap_CA,year,month,week,dayofweek,quarter,is_weekend
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,NaN,0,2011,1,4,5,1,1
1,FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0,2011,1,4,5,1,1
2,FOODS_1_003_CA_1_validation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0,2011,1,4,5,1,1
3,FOODS_1_004_CA_1_validation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,NaN,0,2011,1,4,5,1,1
4,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,NaN,0,2011,1,4,5,1,1


In [6]:
sales = sales.sort_values(['item_id', 'date'])

for lag in [7, 14, 28]:
    sales[f'lag_{lag}'] = sales.groupby('item_id')['sales'].shift(lag)

In [7]:
for window in [7, 14, 28]:
    sales[f'rolling_mean_{window}'] = (
        sales.groupby('item_id')['sales']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

In [8]:
sales['is_event'] = sales['event_name_1'].notna().astype(int)
sales = sales.dropna()
print("Final shape:", sales.shape)

sales.to_csv(r'D:\Naveed\Educational\retail-demand-forecasting\retail-demand-forecasting\data\processed\sales_features.csv', index=False)
print("Saved ✅")

Final shape: (216987, 24)
Saved ✅
